## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import re
import string
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## 2. Load Data

In [2]:
df = pd.read_csv('/content/email_spam_dataset.csv')
print('Shape:', df.shape)
print('\nLabel distribution:')
print(df['label'].value_counts())
df.head()

Shape: (2893, 3)

Label distribution:
label
0    2412
1     481
Name: count, dtype: int64


,subject,message,label
0,job posting - apple-iss research center,content - length : 3386 apple-iss research cen...,0
1,NaN,"lang classification grimes , joseph e . and ba...",0
2,query : letter frequencies for text identifica...,i am posting this inquiry for sergei atamas ( ...,0
3,risk,a colleague and i are researching the differin...,0
4,request book information,earlier this morning i was on the phone with a...,0


## 3. Preprocessing

In [3]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    """Lowercase, remove punctuation, remove stopwords."""
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)                          # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 1]
    return ' '.join(words)

# Fill missing subjects and combine subject + message for richer features
df['subject'] = df['subject'].fillna('')
df['text'] = df['subject'] + ' ' + df['message']

# Apply the same preprocessing to the combined text
df['text_clean'] = df['text'].apply(preprocess)

print('Sample cleaned text:')
print(df['text_clean'].iloc[0][:200])
print('\nNo missing values after fill:', df['text_clean'].isnull().sum())

Sample cleaned text:
job posting appleiss research center content length appleiss research center us million joint venture apple computer inc institute systems science national university singapore located singapore looki

No missing values after fill: 0


## 4. Train / Test Split

In [9]:
X = df['text_clean']
y = df['label']   # integers 0/1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples:', len(X_train))
print('Test samples    :', len(X_test))

Training samples: 2314
Test samples    : 579


## 5. Build Pipeline & Train

In [10]:
#Use a Pipeline so vectorizer + model are always applied togethe this prevents the mismatch between training and prediction

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
        max_features=50000,
        min_df=2
    )),
    ('clf', LogisticRegression(max_iter=1000, random_state=42, C=5.0))
])

pipeline.fit(X_train, y_train)
print('Model trained successfully!')

Model trained successfully!


## 6. Evaluate

In [6]:
y_pred = pipeline.predict(X_test)

print('Test Accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Spam', 'Spam']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Test Accuracy: 98.27 %

Classification Report:
              precision    recall  f1-score   support

    Not Spam       0.98      1.00      0.99       483
        Spam       1.00      0.90      0.95        96

    accuracy                           0.98       579
   macro avg       0.99      0.95      0.97       579
weighted avg       0.98      0.98      0.98       579

Confusion Matrix:
[[483   0]
 [ 10  86]]
